In [4]:
!pip install sentence-transformers chromadb


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
chunks_path = Path("D:/sudhendra/learning projects/GraphRAG/data/apple23chunks.csv")
chunks_df = pd.read_csv(chunks_path)
chunks_df.head()

,page,chunk_id,text
0,1,page_1_chunk_0,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,1,page_1_chunk_1,—\nThe Nasdaq Stock Market LLC\n1.375% Notes d...
2,2,page_2_chunk_0,Indicate by check mark whether the Registrant ...
3,2,page_2_chunk_1,any new or revised financial accounting standa...
4,2,page_2_chunk_2,day of the Registrant’s most recently complete...


In [3]:
len(chunks_df)

239

loading the model 

In [4]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

d:\sudhendra\learning projects\GraphRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11603.28it/s]


In [5]:
test_text = chunks_df.loc[0,"text"]
embedding = embedding_model.encode(test_text)
embedding.shape

(384,)

creating chromaDB collection and storing chunks in it

In [6]:
import chromadb
chroma_client = chromadb.PersistentClient(path = "D:/sudhendra/learning projects/GraphRAG/vector_db")

collection = chroma_client.get_or_create_collection(
    name = "apple23_10k"
)

In [7]:
ids = chunks_df["chunk_id"].astype(str).tolist()
documents = chunks_df["text"].astype(str).tolist()

metadata = [
    {
        "page" : int(row["page"]),
        "chunk_id" : str(row["chunk_id"])
    }
    for _,row in chunks_df.iterrows()
]


In [8]:
embeddings = embedding_model.encode(
    documents,
    batch_size = 32,
    show_progress_bar = True
).tolist()

Batches: 100%|██████████| 8/8 [00:04<00:00,  1.86it/s]


In [9]:
collection.add(
    ids = ids,
    documents = documents,
    embeddings = embeddings,
    metadatas = metadata
)

testing the semantic search

In [10]:
query = "What were Apple's net sales in 2023"

query_embedding = embedding_model.encode(query).tolist()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results = 5
)
results

{'ids': [['page_25_chunk_1',
   'page_23_chunk_1',
   'page_38_chunk_0',
   'page_51_chunk_0',
   'page_11_chunk_8']],
 'embeddings': None,
 'documents': [['iPhone\niPhone net sales decreased 2% or $4.9 billion during 2023 compared to 2022 due to lower net sales of non-Pro iPhone models, \npartially offset by higher net sales of Pro iPhone models.\nMac\nMac net sales decreased 27% or $10.8 billion during 2023 compared to 2022 due primarily to lower net sales of laptops.\niPad\niPad net sales decreased 3% or $1.0 billion during 2023 compared to 2022 due primarily to lower net sales of iPad mini and iPad \nAir, partially offset by the combined net sales of iPad 9th and 10th generation.\nWearables, Home and Accessories\nWearables, Home and Accessories net sales decreased 3% or $1.4 billion during 2023 compared to 2022 due primarily to lower \nnet sales of Wearables and Accessories.\nServices\nServices net sales increased 9% or $7.1 billion during 2023 compared to 2022 due to higher net sa

In [11]:
for i in range(5):
    print("Result:", i + 1)
    print("Page:", results["metadatas"][0][i]["page"])
    print("Distance:", results["distances"][0][i])
    print(results["documents"][0][i][:1000])
    print("-" * 100)

Result: 1
Page: 25
Distance: 0.3937910795211792
iPhone
iPhone net sales decreased 2% or $4.9 billion during 2023 compared to 2022 due to lower net sales of non-Pro iPhone models, 
partially offset by higher net sales of Pro iPhone models.
Mac
Mac net sales decreased 27% or $10.8 billion during 2023 compared to 2022 due primarily to lower net sales of laptops.
iPad
iPad net sales decreased 3% or $1.0 billion during 2023 compared to 2022 due primarily to lower net sales of iPad mini and iPad 
Air, partially offset by the combined net sales of iPad 9th and 10th generation.
Wearables, Home and Accessories
Wearables, Home and Accessories net sales decreased 3% or $1.4 billion during 2023 compared to 2022 due primarily to lower 
net sales of Wearables and Accessories.
Services
Services net sales increased 9% or $7.1 billion during 2023 compared to 2022 due to higher net sales across all lines of 
business.
Apple Inc. | 2023 Form 10-K | 22
-----------------------------------------------------

In [12]:
queries = [
    "What are Apple's main products?",
    "What risks does Apple mention?",
    "How much revenue did Apple generate in 2023?",
    "What is Apple's research and development expense?",
    "What are Apple's business segments?"
]

In [13]:
for query in queries:
    print("\nQUERY:", query)
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results = 5
    )

    for i in range(3):
        print("Page:", results["metadatas"][0][i]["page"])
        print(results["documents"][0][i][:500])
        print("-" * 80)


QUERY: What are Apple's main products?
Page: 12
products. The Company’s products and operating systems are subject to rapid technological change, and when third-party 
developers are unable to or choose not to keep up with this pace of change, their applications can fail to take advantage of these 
changes to deliver improved customer experiences, can operate incorrectly, and can result in dissatisfied customers and lower 
customer demand for the Company’s products.
Apple Inc. | 2023 Form 10-K | 9
--------------------------------------------------------------------------------
Page: 4
Mac
Mac® is the Company’s line of personal computers based on its macOS® operating system. The Mac line includes laptops 
MacBook Air® and MacBook Pro®, as well as desktops iMac®, Mac mini®, Mac Studio® and Mac Pro®.
iPad
iPad® is the Company’s line of multipurpose tablets based on its iPadOS® operating system. The iPad line includes iPad Pro®, 
iPad Air®, iPad and iPad mini®.
Wearables, Home and Accesso